In [1]:
import pandas as pd
import re
import os
import sys
import torch
from pathlib import Path

In [ ]:
data = pd.read_csv('/home/your address/shenzhen/6深圳_携程图像文本/0深圳_携程text/2深圳市盐田区大梅沙海滨公园.csv', encoding='utf-8')
#change the address above to your own address
data.head()

,用户名,原始用户名,评论,时间,评分,图像数量,图像文件名列表
0,2深圳市盐田区大梅沙海滨公园_M870550,M87****0550,"沙滩上人山人海，随便一躺就能和陌生人组成""人体沙画""；海浪一来，拖鞋比主人跑得还快，建议提前...",2025-04-05IP属地：广东,5分 超棒,4,2深圳市盐田区大梅沙海滨公园_M870550_1.jpg;2深圳市盐田区大梅沙海滨公园_M8...
1,2深圳市盐田区大梅沙海滨公园_旋转的紫荆花,旋转的紫荆花,梅沙湾风景不错，山青水秀，地铁🚇八号线直达。站到海边🏝️🏝️顿时心胸开阔，海涛阵阵生生不息。...,2025-01-12IP属地：广东,4分 满意,5,2深圳市盐田区大梅沙海滨公园_旋转的紫荆花_1.jpg;2深圳市盐田区大梅沙海滨公园_旋转的...
2,2深圳市盐田区大梅沙海滨公园_206566,206****566,大梅沙海滨公园（Dameisha Park），是一个集休闲度假、观光旅游、运动娱乐为一体的公...,2024-12-13IP属地：江苏,5分 超棒,3,2深圳市盐田区大梅沙海滨公园_206566_1.jpg;2深圳市盐田区大梅沙海滨公园_206...
3,2深圳市盐田区大梅沙海滨公园_南巫扶摇,南巫扶摇,⛰【大梅沙海滨沙滩景点攻略】\n📍详细地址：深圳市盐田区梅沙街道梅沙社区盐梅路\n🚗交通攻略...,2020-05-25IP属地：未知,5分 超棒,5,2深圳市盐田区大梅沙海滨公园_南巫扶摇_1.jpg;2深圳市盐田区大梅沙海滨公园_南巫扶摇_...
4,2深圳市盐田区大梅沙海滨公园_雪0106,雪0106,大梅沙海滨公园很漂亮，交通便捷，附近有很多公交车站，可以乘公交到地铁站。旁边是大梅沙奥特莱斯...,2019-07-10IP属地：未知,5分 超棒,15,2深圳市盐田区大梅沙海滨公园_雪0106_1.jpg;2深圳市盐田区大梅沙海滨公园_雪010...


### Dataset Column Definitions & Content Logic

This dataset contains user reviews and metadata for "Dameisha Seaside Park" (大梅沙海滨公园) in Shenzhen. Below is the explanation for each column based on standard travel data structures:

| Column Name | English Translation | Definition & Content Logic |
| :--- | :--- | :--- |
| **用户名** | **Username** | A composite identifier combining the location name and the original username. This string serves as the specific prefix for naming image files associated with the review. |
| **原始用户名** | **Original Username** | The actual username of the reviewer. Note that for privacy, long numeric IDs (like phone numbers) have their middle four digits masked with asterisks (*), making them distinct from the composite 'Username'. Standard names remain unmasked. |
| **评论** | **Comment** | The actual text content of the user's review describing their experience, feelings, and observations. This is the **primary input** for our sentiment analysis model. |
| **时间** | **Time and IP Location** | Contains both the timestamp of the review and the user's IP location (e.g., "Guangdong"). This is useful for analyzing temporal trends and analyzing tourist vs. local sentiment patterns. |
| **评分** | **Rating** | The numerical rating given by the user (typically on a 1-5 scale). While it serves as a baseline, it may not always perfectly reflect the nuanced sentiment in the text. |
| **图片数量** | **Image Count** | The total number of images uploaded with this review. |
| **图片文件名列表** | **Image List** | A list containing the filenames of all images associated with this review, corresponding to the 'Username' prefix. |

In [5]:
import sys
import os
import torch
from pathlib import Path

# Robustly add src to path (handling notebook environment where __file__ is generic)
current_dir = os.getcwd()
src_path = os.path.abspath(os.path.join(current_dir, '..', 'src'))
if src_path not in sys.path:
    sys.path.append(src_path)

from models.llm_analyzer import PsychologicalStateAnalyzer

# Initialize with updated analyzer that uses Dataset internally
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
analyzer = PsychologicalStateAnalyzer(device=device)

# Filter out non-string rows to avoid errors
valid_texts_mask = data['评论'].apply(lambda x: isinstance(x, str) and len(x.strip()) > 0)
texts_to_process = data.loc[valid_texts_mask, '评论'].tolist()

print(f"Total valid texts to process: {len(texts_to_process)}")

# Run batch processing directly via the class method
try:
    if texts_to_process:
        # Calls the updated analyze_sentiment which now uses TextDataset and proper pipeline batching
        print("Starting batch analysis...")
        results = analyzer.analyze_sentiment(texts_to_process, batch_size=64)
        
        # Extract just the scores
        sentiment_scores = [r['score'] for r in results]
        
        # Assign back to DataFrame
        data.loc[valid_texts_mask, 'sentiment_score'] = sentiment_scores
        
        # Fill NaN for invalid/empty texts
        data['sentiment_score'] = data['sentiment_score'].fillna(0.5) 
        
        print("Processing complete.")
    else:
        print("No valid texts to process.")
        
except Exception as e:
    print(f"An error occurred: {e}")

data[['评论', 'sentiment_score']].head(10)

Using device: cuda
Initializing PsychologicalStateAnalyzer on cuda...


Device set to use cuda:0


Total valid texts to process: 1160
Starting batch analysis...
Processing complete.


,评论,sentiment_score
0,"沙滩上人山人海，随便一躺就能和陌生人组成""人体沙画""；海浪一来，拖鞋比主人跑得还快，建议提前...",0.007515
1,梅沙湾风景不错，山青水秀，地铁🚇八号线直达。站到海边🏝️🏝️顿时心胸开阔，海涛阵阵生生不息。...,0.816329
2,大梅沙海滨公园（Dameisha Park），是一个集休闲度假、观光旅游、运动娱乐为一体的公...,0.897180
3,⛰【大梅沙海滨沙滩景点攻略】\n📍详细地址：深圳市盐田区梅沙街道梅沙社区盐梅路\n🚗交通攻略...,0.601739
4,大梅沙海滨公园很漂亮，交通便捷，附近有很多公交车站，可以乘公交到地铁站。旁边是大梅沙奥特莱斯...,0.982684
5,大梅沙海滨公园：深圳的蓝色梦幻乐园\n\n在繁华喧嚣的深圳，有一片能让人瞬间忘却烦恼的蓝色天...,0.908680
6,深圳大梅沙是一个极具魅力的海滨旅游胜地。\n\n1. 优点\n 深圳大梅沙拥...,0.979518
7,大梅沙海滨公园位于深圳特区东部，地处南海之滨。这里三面环山，一面临海，中间绵延1800米的开...,0.831596
8,深圳人真是有福气，乘个地铁就能来到大梅沙沙滩上，吹吹海风，有老年人在这里游泳，这里配备有厕所...,0.950379
9,大梅沙海滨公园位于盐田港与小梅沙之间，设施完善，景色旖旎。入园是五颜六色、形态各异的“鸟人”...,0.511006


In [4]:
data.to_csv('2深圳市盐田区大梅沙海滨公园_with_sentiment.csv', index=False, encoding='utf-8')

## Example for Batch Processing Multiple CSV Files to generate scores 

To process multiple CSV files in a folder, you can use the following script logic:

```python
import glob
import os
import pandas as pd
from models.llm_analyzer import PsychologicalStateAnalyzer

# 1. Initialize the model (do this only once)
analyzer = PsychologicalStateAnalyzer(device="cuda")

# 2. Get all CSV file paths
csv_files = glob.glob('shenzhen/**/*.csv', recursive=True)

# 3. Loop through files
for file_path in csv_files:
    print(f"Processing {file_path}...")
    
    try:
        # Read Data
        df = pd.read_csv(file_path, encoding='utf-8')
        
        # Check for '评论' (Comment) column
        if '评论' not in df.columns:
            print(f"Skipping {file_path}: No '评论' column found.")
            continue
            
        # Prepare text data
        valid_mask = df['评论'].apply(lambda x: isinstance(x, str) and len(x.strip()) > 0)
        texts = df.loc[valid_mask, '评论'].tolist()
        
        if not texts:
            continue
            
        # Batch Analysis (Core acceleration step)
        # analyze_sentiment internally uses TextDataset and GPU batching automatically
        results = analyzer.analyze_sentiment(texts, batch_size=64) # Increase batch_size based on VRAM
        scores = [r['score'] for r in results]
        
        # Save results
        df.loc[valid_mask, 'sentiment_score'] = scores
        df['sentiment_score'] = df['sentiment_score'].fillna(0.5)
        
        # Construct output path (e.g., save to a processed_data folder)
        output_path = file_path.replace('.csv', '_analyzed.csv')
        df.to_csv(output_path, index=False, encoding='utf-8')
        print(f"Saved to {output_path}")
        
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
```